In [1]:
# ========================================================
# Joint MNIST + CIFAR FL Poisoning Attack Detector
# Domain-aware, Label-aware, DataFrame-safe PKL
# ========================================================

import pandas as pd
import numpy as np
import joblib

from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

# --------------------------------------------------------
# 1. Load BOTH datasets
# --------------------------------------------------------
print(" Loading datasets...")

DATA_PATH_MNIST = "finaltrial_mnist.csv"
DATA_PATH_CIFAR = "3finaltrial_cifar10.csv"

df_mnist = pd.read_csv(DATA_PATH_MNIST)
df_cifar = pd.read_csv(DATA_PATH_CIFAR)

# Add dataset identifier
df_mnist["dataset_source"] = "mnist"
df_cifar["dataset_source"] = "cifar"

# Combine in memory
df = pd.concat([df_mnist, df_cifar], ignore_index=True)

print(f" Combined dataset shape: {df.shape}")

# --------------------------------------------------------
# 2. Feature Engineering
# --------------------------------------------------------
print("🔧 Preparing features...")

# Pure numeric features
pure_numeric_features = [col for col in df.columns if col.startswith("dct_")]

# Categorical features (INCLUDING domain)
categorical_features = [
    "attack_variant",
    "client_classes",
    "dataset_source"
]

# Target
y = df["label"].astype(int)

# Full feature DataFrame (this is what PKL will expect)
X = df[pure_numeric_features + categorical_features]

# --------------------------------------------------------
# 3. Preprocessing Pipeline
# --------------------------------------------------------
numeric_transformer = StandardScaler()

categorical_transformer = OneHotEncoder(
    handle_unknown="ignore"
)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, pure_numeric_features),
        ("cat", categorical_transformer, categorical_features),
    ]
)

# --------------------------------------------------------
# 4. Train-Test Split (Label + Domain Aware)
# --------------------------------------------------------
df["stratify_key"] = (
    df["label"].astype(str) + "_" + df["dataset_source"]
)

print(" Splitting train-test (joint stratification)...")

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.25,
    random_state=42,
    stratify=df["stratify_key"]
)

print(f"Train shape: {X_train.shape}, Test shape: {X_test.shape}")
print("Train domain distribution:")
print(X_train["dataset_source"].value_counts())

# --------------------------------------------------------
# 5. Random Forest Model
# --------------------------------------------------------
print("\n🌲 Training RandomForestClassifier...")

rf_model = Pipeline(steps=[
    ("preprocess", preprocessor),
    ("clf", RandomForestClassifier(
        n_estimators=400,
        max_depth=None,
        class_weight="balanced",
        n_jobs=-1,
        random_state=42
    ))
])

rf_model.fit(X_train, y_train)

rf_preds = rf_model.predict(X_test)

print("\n RandomForest Results:")
print(classification_report(y_test, rf_preds))
print("Confusion Matrix:\n", confusion_matrix(y_test, rf_preds))

# Save RF bundle (model + schema)
rf_bundle = {
    "model": rf_model,
    "features": pure_numeric_features + categorical_features
}

#joblib.dump(rf_bundle, "combined_fl_attack_detector_rf.pkl")
#print(" Saved: combined_fl_attack_detector_rf.pkl")


print("\n🚀 Training XGBoost...")

xgb_model = Pipeline(steps=[
    ("preprocess", preprocessor),
    ("clf", XGBClassifier(
        objective="multi:softmax",
        num_class=len(np.unique(y)),
        n_estimators=400,
        max_depth=None,
        learning_rate=0.1,
        subsample=0.8,
        colsample_bytree=0.8,
        eval_metric="mlogloss",
        tree_method="hist",
        random_state=42
    ))
])

xgb_model.fit(X_train, y_train)

xgb_preds = xgb_model.predict(X_test)

print("\n XGBoost Results:")
print(classification_report(y_test, xgb_preds))
print("Confusion Matrix:\n", confusion_matrix(y_test, xgb_preds))


from sklearn.svm import SVC

print("\n🔹 Training SVM (RBF)...")

svm_model = Pipeline(steps=[
    ("preprocess", preprocessor),
    ("clf", SVC(
        kernel="rbf",
        C=10,
        gamma="scale",
        class_weight="balanced",
        probability=False,
        random_state=42
    ))
])

svm_model.fit(X_train, y_train)

svm_preds = svm_model.predict(X_test)

print("\n SVM (RBF) Results:")
print(classification_report(y_test, svm_preds))
print("Confusion Matrix:\n", confusion_matrix(y_test, svm_preds))


from sklearn.neural_network import MLPClassifier

print("\n🧠 Training MLP...")

mlp_model = Pipeline(steps=[
    ("preprocess", preprocessor),
    ("clf", MLPClassifier(
        hidden_layer_sizes=(256, 128),
        activation="relu",
        solver="adam",
        alpha=1e-4,
        batch_size=128,
        learning_rate="adaptive",
        max_iter=200,
        random_state=42
    ))
])

mlp_model.fit(X_train, y_train)

mlp_preds = mlp_model.predict(X_test)

print("\n MLP Results:")
print(classification_report(y_test, mlp_preds))
print("Confusion Matrix:\n", confusion_matrix(y_test, mlp_preds))




 Loading datasets...
 Combined dataset shape: (10000, 327)
🔧 Preparing features...
 Splitting train-test (joint stratification)...
Train shape: (7500, 303), Test shape: (2500, 303)
Train domain distribution:
dataset_source
mnist    3750
cifar    3750
Name: count, dtype: int64

🌲 Training RandomForestClassifier...

 RandomForest Results:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00      1750
           1       0.66      0.64      0.65       153
           2       1.00      1.00      1.00       149
           3       1.00      1.00      1.00       145
           4       0.65      0.65      0.65       150
           5       1.00      1.00      1.00       153

    accuracy                           0.96      2500
   macro avg       0.88      0.88      0.88      2500
weighted avg       0.96      0.96      0.96      2500

Confusion Matrix:
 [[1750    0    0    0    0    0]
 [   3   98    0    0   52    0]
 [   0    0  149    0    0    0]


In [2]:
from sklearn.neighbors import KNeighborsClassifier

print("\n📍 Training KNN...")

knn_model = Pipeline(steps=[
    ("preprocess", preprocessor),
    ("clf", KNeighborsClassifier(
        n_neighbors=15,        # equivalent role to n_estimators
        weights="distance",    # better for attack boundaries
        metric="minkowski",
        p=2                    # Euclidean distance
    ))
])

knn_model.fit(X_train, y_train)

knn_preds = knn_model.predict(X_test)

print("\n KNN Results:")
print(classification_report(y_test, knn_preds))
print("Confusion Matrix:\n", confusion_matrix(y_test, knn_preds))



📍 Training KNN...

 KNN Results:
              precision    recall  f1-score   support

           0       0.98      1.00      0.99      1750
           1       0.62      0.84      0.71       153
           2       1.00      1.00      1.00       149
           3       1.00      1.00      1.00       145
           4       0.97      0.39      0.56       150
           5       1.00      1.00      1.00       153

    accuracy                           0.95      2500
   macro avg       0.93      0.87      0.88      2500
weighted avg       0.96      0.95      0.95      2500

Confusion Matrix:
 [[1750    0    0    0    0    0]
 [  22  129    0    0    2    0]
 [   0    0  149    0    0    0]
 [   0    0    0  145    0    0]
 [  11   80    0    0   59    0]
 [   0    0    0    0    0  153]]


In [3]:
joblib.dump( mlp_model, "combined_mlp_model.pkl")
print(" Saved: combined_mlp_model.pkl")

 Saved: combined_mlp_model.pkl


training on just mnist


In [4]:
# ========================================================
# MNIST-only Training → Cross-Dataset Generalization Test
# Classifier: SVM (RBF)
# ========================================================

import pandas as pd
import numpy as np

from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.svm import SVC

# --------------------------------------------------------
# 1. Load MNIST Dataset (Training Domain)
# --------------------------------------------------------
DATA_PATH_MNIST = "finaltrial_mnist.csv"
DATA_PATH_CIFAR = "3finaltrial_cifar10.csv"

print("🔹 Loading MNIST dataset...")
df_mnist = pd.read_csv(DATA_PATH_MNIST)

# --------------------------------------------------------
# 2. Feature Engineering
# --------------------------------------------------------
print("🔧 Preparing features...")

# DCT features
numeric_features = [col for col in df_mnist.columns if col.startswith("dct_")]

# Metadata features (NO domain label → fair generalization)
categorical_features = ["attack_variant", "client_classes"]

# Target
y = df_mnist["label"].astype(int)

# Feature matrix
X = df_mnist[numeric_features + categorical_features]

# --------------------------------------------------------
# 3. Preprocessing Pipeline
# --------------------------------------------------------
numeric_transformer = StandardScaler()

categorical_transformer = OneHotEncoder(
    handle_unknown="ignore"
)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features),
    ]
)

# --------------------------------------------------------
# 4. Train-Test Split (MNIST only)
# --------------------------------------------------------
print("✂️ Splitting MNIST train-test...")

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=42,
    stratify=y
)

# --------------------------------------------------------
# 5. SVM (RBF) Model
# --------------------------------------------------------
print("\n🔹 Training SVM (RBF) on MNIST only...")

svm_model = Pipeline(steps=[
    ("preprocess", preprocessor),
    ("clf", SVC(
        kernel="rbf",
        C=10,
        gamma="scale",
        class_weight="balanced",
        decision_function_shape="ovr",
        random_state=42
    ))
])

svm_model.fit(X_train, y_train)

# --------------------------------------------------------
# 6. Evaluation A: MNIST → MNIST Test
# --------------------------------------------------------
print("\n📊 MNIST → MNIST Test Results")

mnist_preds = svm_model.predict(X_test)

print(classification_report(y_test, mnist_preds))
print("Confusion Matrix:\n", confusion_matrix(y_test, mnist_preds))

# --------------------------------------------------------
# 7. Evaluation B: MNIST → CIFAR-10 (OOD Test)
# --------------------------------------------------------
print("\n🚨 MNIST → CIFAR-10 Generalization Results")

print("🔄 Loading CIFAR-10 dataset...")
df_cifar = pd.read_csv(DATA_PATH_CIFAR)

# Ensure SAME feature schema
X_cifar = df_cifar[numeric_features + categorical_features]
y_cifar = df_cifar["label"].astype(int)

cifar_preds = svm_model.predict(X_cifar)

print(classification_report(y_cifar, cifar_preds))
print("Confusion Matrix:\n", confusion_matrix(y_cifar, cifar_preds))


🔹 Loading MNIST dataset...
🔧 Preparing features...
✂️ Splitting MNIST train-test...

🔹 Training SVM (RBF) on MNIST only...

📊 MNIST → MNIST Test Results
              precision    recall  f1-score   support

           0       1.00      1.00      1.00       875
           1       0.77      0.66      0.71        80
           2       1.00      1.00      1.00        73
           3       1.00      1.00      1.00        68
           4       0.69      0.79      0.73        75
           5       1.00      1.00      1.00        79

    accuracy                           0.97      1250
   macro avg       0.91      0.91      0.91      1250
weighted avg       0.97      0.97      0.97      1250

Confusion Matrix:
 [[875   0   0   0   0   0]
 [  0  53   0   0  27   0]
 [  0   0  73   0   0   0]
 [  0   0   0  68   0   0]
 [  0  16   0   0  59   0]
 [  0   0   0   0   0  79]]

🚨 MNIST → CIFAR-10 Generalization Results
🔄 Loading CIFAR-10 dataset...
              precision    recall  f1-score   sup

In [5]:
# ========================================================
# MNIST-only Training → Cross-Dataset Generalization Test
# Classifier: SVM (RBF)
# ========================================================

import pandas as pd
import numpy as np

from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.svm import SVC

# --------------------------------------------------------
# 1. Load MNIST Dataset (Training Domain)
# --------------------------------------------------------
DATA_PATH_MNIST = "finaltrial_mnist.csv"
DATA_PATH_CIFAR = "3finaltrial_cifar10.csv"

print("🔹 Loading CIFAR dataset...")
df_mnist = pd.read_csv(DATA_PATH_CIFAR)

# --------------------------------------------------------
# 2. Feature Engineering
# --------------------------------------------------------
print("🔧 Preparing features...")

# DCT features
numeric_features = [col for col in df_mnist.columns if col.startswith("dct_")]

# Metadata features (NO domain label → fair generalization)
categorical_features = ["attack_variant", "client_classes"]

# Target
y = df_mnist["label"].astype(int)

# Feature matrix
X = df_mnist[numeric_features + categorical_features]

# --------------------------------------------------------
# 3. Preprocessing Pipeline
# --------------------------------------------------------
numeric_transformer = StandardScaler()

categorical_transformer = OneHotEncoder(
    handle_unknown="ignore"
)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features),
    ]
)

# --------------------------------------------------------
# 4. Train-Test Split (MNIST only)
# --------------------------------------------------------
print("✂️ Splitting train-test...")

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=42,
    stratify=y
)

# --------------------------------------------------------
# 5. SVM (RBF) Model
# --------------------------------------------------------
print("\n🔹 Training SVM (RBF)...")

svm_model = Pipeline(steps=[
    ("preprocess", preprocessor),
    ("clf", SVC(
        kernel="rbf",
        C=10,
        gamma="scale",
        class_weight="balanced",
        decision_function_shape="ovr",
        random_state=42
    ))
])

svm_model.fit(X_train, y_train)

# --------------------------------------------------------
# 6. Evaluation A: MNIST → MNIST Test
# --------------------------------------------------------

mnist_preds = svm_model.predict(X_test)

print(classification_report(y_test, mnist_preds))
print("Confusion Matrix:\n", confusion_matrix(y_test, mnist_preds))

# --------------------------------------------------------
# 7. Evaluation B: MNIST → CIFAR-10 (OOD Test)
# --------------------------------------------------------


print("🔄 Loading MNIST dataset...")
df_cifar = pd.read_csv(DATA_PATH_MNIST)

# Ensure SAME feature schema
X_cifar = df_cifar[numeric_features + categorical_features]
y_cifar = df_cifar["label"].astype(int)

cifar_preds = svm_model.predict(X_cifar)

print(classification_report(y_cifar, cifar_preds))
print("Confusion Matrix:\n", confusion_matrix(y_cifar, cifar_preds))


🔹 Loading CIFAR dataset...
🔧 Preparing features...
✂️ Splitting train-test...

🔹 Training SVM (RBF)...
              precision    recall  f1-score   support

           0       1.00      1.00      1.00       875
           1       0.88      0.79      0.83        73
           2       1.00      1.00      1.00        76
           3       1.00      1.00      1.00        77
           4       0.82      0.89      0.85        75
           5       1.00      1.00      1.00        74

    accuracy                           0.98      1250
   macro avg       0.95      0.95      0.95      1250
weighted avg       0.98      0.98      0.98      1250

Confusion Matrix:
 [[875   0   0   0   0   0]
 [  0  58   0   0  15   0]
 [  0   0  76   0   0   0]
 [  0   0   0  77   0   0]
 [  0   8   0   0  67   0]
 [  0   0   0   0   0  74]]
🔄 Loading MNIST dataset...
              precision    recall  f1-score   support

           0       1.00      1.00      1.00      3500
           1       0.80      0.22   

In [6]:
# ========================================================
# MNIST-only Training → Cross-Dataset Generalization Test
# Classifier: SVM (RBF)
# ========================================================

import pandas as pd
import numpy as np

from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.svm import SVC

# --------------------------------------------------------
# 1. Load MNIST Dataset (Training Domain)
# --------------------------------------------------------
DATA_PATH_MNIST = "finaltrial_mnist.csv"
DATA_PATH_CIFAR = "3finaltrial_cifar10.csv"

print("🔹 Loading MNIST dataset...")
df_mnist = pd.read_csv(DATA_PATH_MNIST)

# --------------------------------------------------------
# 2. Feature Engineering
# --------------------------------------------------------
print("🔧 Preparing features...")

# DCT features
numeric_features = [col for col in df_mnist.columns if col.startswith("dct_")]

# Metadata features (NO domain label → fair generalization)
categorical_features = ["attack_variant", "client_classes"]

# Target
y = df_mnist["label"].astype(int)

# Feature matrix
X = df_mnist[numeric_features + categorical_features]

# --------------------------------------------------------
# 3. Preprocessing Pipeline
# --------------------------------------------------------
numeric_transformer = StandardScaler()

categorical_transformer = OneHotEncoder(
    handle_unknown="ignore"
)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features),
    ]
)

# --------------------------------------------------------
# 4. Train-Test Split (MNIST only)
# --------------------------------------------------------
print("✂️ Splitting MNIST train-test...")

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=42,
    stratify=y
)

# --------------------------------------------------------
# 5. SVM (RBF) Model
# --------------------------------------------------------


rf_model = Pipeline(steps=[
    ("preprocess", preprocessor),
    ("clf", RandomForestClassifier(
        n_estimators=400,
        max_depth=None,
        class_weight="balanced",
        n_jobs=-1,
        random_state=42
    ))
])

rf_model.fit(X_train, y_train)

rf_preds = rf_model.predict(X_test)
print("\n📊 MNIST → MNIST Test Results")
print("\n RandomForest Results:")
print(classification_report(y_test, rf_preds))
print("Confusion Matrix:\n", confusion_matrix(y_test, rf_preds))




# --------------------------------------------------------
# 7. Evaluation B: MNIST → CIFAR-10 (OOD Test)
# --------------------------------------------------------
print("\n🚨 MNIST → CIFAR-10 Generalization Results")

print("🔄 Loading CIFAR-10 dataset...")
df_cifar = pd.read_csv(DATA_PATH_CIFAR)

# Ensure SAME feature schema
X_cifar = df_cifar[numeric_features + categorical_features]
y_cifar = df_cifar["label"].astype(int)

cifar_preds = rf_model.predict(X_cifar)

print(classification_report(y_cifar, cifar_preds))
print("Confusion Matrix:\n", confusion_matrix(y_cifar, cifar_preds))


🔹 Loading MNIST dataset...
🔧 Preparing features...
✂️ Splitting MNIST train-test...

📊 MNIST → MNIST Test Results

 RandomForest Results:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00       875
           1       0.70      0.68      0.69        80
           2       1.00      1.00      1.00        73
           3       0.97      1.00      0.99        68
           4       0.66      0.65      0.66        75
           5       1.00      1.00      1.00        79

    accuracy                           0.96      1250
   macro avg       0.89      0.89      0.89      1250
weighted avg       0.96      0.96      0.96      1250

Confusion Matrix:
 [[875   0   0   0   0   0]
 [  1  54   0   0  25   0]
 [  0   0  73   0   0   0]
 [  0   0   0  68   0   0]
 [  1  23   0   2  49   0]
 [  0   0   0   0   0  79]]

🚨 MNIST → CIFAR-10 Generalization Results
🔄 Loading CIFAR-10 dataset...
              precision    recall  f1-score   support

         

In [7]:
# ========================================================
# MNIST-only Training → Cross-Dataset Generalization Test
# Classifier: SVM (RBF)
# ========================================================

import pandas as pd
import numpy as np

from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.svm import SVC

# --------------------------------------------------------
# 1. Load MNIST Dataset (Training Domain)
# --------------------------------------------------------
DATA_PATH_MNIST = "finaltrial_mnist.csv"
DATA_PATH_CIFAR = "3finaltrial_cifar10.csv"

print("🔹 Loading CIFAR dataset...")
df_mnist = pd.read_csv(DATA_PATH_CIFAR)

# --------------------------------------------------------
# 2. Feature Engineering
# --------------------------------------------------------
print("🔧 Preparing features...")

# DCT features
numeric_features = [col for col in df_mnist.columns if col.startswith("dct_")]

# Metadata features (NO domain label → fair generalization)
categorical_features = ["attack_variant", "client_classes"]

# Target
y = df_mnist["label"].astype(int)

# Feature matrix
X = df_mnist[numeric_features + categorical_features]

# --------------------------------------------------------
# 3. Preprocessing Pipeline
# --------------------------------------------------------
numeric_transformer = StandardScaler()

categorical_transformer = OneHotEncoder(
    handle_unknown="ignore"
)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features),
    ]
)

# --------------------------------------------------------
# 4. Train-Test Split (MNIST only)
# --------------------------------------------------------
print("✂️ Splitting CIFAR train-test...")

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=42,
    stratify=y
)

# --------------------------------------------------------
# 5. SVM (RBF) Model
# --------------------------------------------------------


rf_model = Pipeline(steps=[
    ("preprocess", preprocessor),
    ("clf", RandomForestClassifier(
        n_estimators=400,
        max_depth=None,
        class_weight="balanced",
        n_jobs=-1,
        random_state=42
    ))
])

rf_model.fit(X_train, y_train)

rf_preds = rf_model.predict(X_test)

print("\n RandomForest Results:")
print(classification_report(y_test, rf_preds))
print("Confusion Matrix:\n", confusion_matrix(y_test, rf_preds))


# --------------------------------------------------------
# 6. Evaluation A: MNIST → MNIST Test
# --------------------------------------------------------


# --------------------------------------------------------
# 7. Evaluation B: MNIST → CIFAR-10 (OOD Test)
# --------------------------------------------------------


print("🔄 Loading MNIST dataset...")
df_cifar = pd.read_csv(DATA_PATH_MNIST)

# Ensure SAME feature schema
X_cifar = df_cifar[numeric_features + categorical_features]
y_cifar = df_cifar["label"].astype(int)

cifar_preds = rf_model.predict(X_cifar)

print(classification_report(y_cifar, cifar_preds))
print("Confusion Matrix:\n", confusion_matrix(y_cifar, cifar_preds))


🔹 Loading CIFAR dataset...
🔧 Preparing features...
✂️ Splitting CIFAR train-test...

 RandomForest Results:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00       875
           1       0.60      0.48      0.53        73
           2       1.00      1.00      1.00        76
           3       1.00      1.00      1.00        77
           4       0.60      0.69      0.64        75
           5       1.00      1.00      1.00        74

    accuracy                           0.95      1250
   macro avg       0.87      0.86      0.86      1250
weighted avg       0.95      0.95      0.95      1250

Confusion Matrix:
 [[875   0   0   0   0   0]
 [  3  35   0   0  35   0]
 [  0   0  76   0   0   0]
 [  0   0   0  77   0   0]
 [  0  23   0   0  52   0]
 [  0   0   0   0   0  74]]
🔄 Loading MNIST dataset...
              precision    recall  f1-score   support

           0       1.00      1.00      1.00      3500
           1       0.58      0.

In [10]:
# ========================================================
# MNIST-only Training → Cross-Dataset Generalization Test
# Classifier: SVM (RBF)
# ========================================================

import pandas as pd
import numpy as np

from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.svm import SVC

# --------------------------------------------------------
# 1. Load MNIST Dataset (Training Domain)
# --------------------------------------------------------
DATA_PATH_MNIST = "finaltrial_mnist.csv"
DATA_PATH_CIFAR = "3finaltrial_cifar10.csv"

print("🔹 Loading MNIST dataset...")
df_mnist = pd.read_csv(DATA_PATH_MNIST)

# --------------------------------------------------------
# 2. Feature Engineering
# --------------------------------------------------------
print("🔧 Preparing features...")

# DCT features
numeric_features = [col for col in df_mnist.columns if col.startswith("dct_")]

# Metadata features (NO domain label → fair generalization)
categorical_features = ["attack_variant", "client_classes"]

# Target
y = df_mnist["label"].astype(int)

# Feature matrix
X = df_mnist[numeric_features + categorical_features]

# --------------------------------------------------------
# 3. Preprocessing Pipeline
# --------------------------------------------------------
numeric_transformer = StandardScaler()

categorical_transformer = OneHotEncoder(
    handle_unknown="ignore"
)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features),
    ]
)

# --------------------------------------------------------
# 4. Train-Test Split (MNIST only)
# --------------------------------------------------------
print("✂️ Splitting MNIST train-test...")

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=42,
    stratify=y
)

# --------------------------------------------------------
# 5. SVM (RBF) Model
# --------------------------------------------------------
from sklearn.neural_network import MLPClassifier

print("\n🧠 Training MLP...")

mlp_model = Pipeline(steps=[
    ("preprocess", preprocessor),
    ("clf", MLPClassifier(
        hidden_layer_sizes=(256, 128),
        activation="relu",
        solver="adam",
        alpha=1e-4,
        batch_size=128,
        learning_rate="adaptive",
        max_iter=200,
        random_state=42
    ))
])

mlp_model.fit(X_train, y_train)

mlp_preds = mlp_model.predict(X_test)

print("\n MLP Results:")
print(classification_report(y_test, mlp_preds))
print("Confusion Matrix:\n", confusion_matrix(y_test, mlp_preds))


# --------------------------------------------------------
# 6. Evaluation A: MNIST → MNIST Test
# --------------------------------------------------------


# --------------------------------------------------------
# 7. Evaluation B: MNIST → CIFAR-10 (OOD Test)
# --------------------------------------------------------
print("\n🚨 MNIST → CIFAR-10 Generalization Results")

print("🔄 Loading CIFAR-10 dataset...")
df_cifar = pd.read_csv(DATA_PATH_CIFAR)

# Ensure SAME feature schema
X_cifar = df_cifar[numeric_features + categorical_features]
y_cifar = df_cifar["label"].astype(int)

cifar_preds =  mlp_model.predict(X_cifar)

print(classification_report(y_cifar, cifar_preds))
print("Confusion Matrix:\n", confusion_matrix(y_cifar, cifar_preds))


🔹 Loading MNIST dataset...
🔧 Preparing features...
✂️ Splitting MNIST train-test...

🧠 Training MLP...

 MLP Results:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00       875
           1       1.00      0.97      0.99        80
           2       1.00      1.00      1.00        73
           3       1.00      1.00      1.00        68
           4       0.97      1.00      0.99        75
           5       1.00      1.00      1.00        79

    accuracy                           1.00      1250
   macro avg       1.00      1.00      1.00      1250
weighted avg       1.00      1.00      1.00      1250

Confusion Matrix:
 [[875   0   0   0   0   0]
 [  0  78   0   0   2   0]
 [  0   0  73   0   0   0]
 [  0   0   0  68   0   0]
 [  0   0   0   0  75   0]
 [  0   0   0   0   0  79]]

🚨 MNIST → CIFAR-10 Generalization Results
🔄 Loading CIFAR-10 dataset...
              precision    recall  f1-score   support

           0       1.00      

In [11]:
# ========================================================
# MNIST-only Training → Cross-Dataset Generalization Test
# Classifier: SVM (RBF)
# ========================================================

import pandas as pd
import numpy as np

from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.svm import SVC

# --------------------------------------------------------
# 1. Load MNIST Dataset (Training Domain)
# --------------------------------------------------------
DATA_PATH_MNIST = "finaltrial_mnist.csv"
DATA_PATH_CIFAR = "3finaltrial_cifar10.csv"

print("🔹 Loading CIFAR dataset...")
df_mnist = pd.read_csv(DATA_PATH_CIFAR)

# --------------------------------------------------------
# 2. Feature Engineering
# --------------------------------------------------------
print("🔧 Preparing features...")

# DCT features
numeric_features = [col for col in df_mnist.columns if col.startswith("dct_")]

# Metadata features (NO domain label → fair generalization)
categorical_features = ["attack_variant", "client_classes"]

# Target
y = df_mnist["label"].astype(int)

# Feature matrix
X = df_mnist[numeric_features + categorical_features]

# --------------------------------------------------------
# 3. Preprocessing Pipeline
# --------------------------------------------------------
numeric_transformer = StandardScaler()

categorical_transformer = OneHotEncoder(
    handle_unknown="ignore"
)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features),
    ]
)

# --------------------------------------------------------
# 4. Train-Test Split (MNIST only)
# --------------------------------------------------------
print("✂️ Splitting train-test...")

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=42,
    stratify=y
)

# --------------------------------------------------------
# 5. SVM (RBF) Model
# --------------------------------------------------------
from sklearn.neural_network import MLPClassifier

print("\n🧠 Training MLP...")

mlp_model = Pipeline(steps=[
    ("preprocess", preprocessor),
    ("clf", MLPClassifier(
        hidden_layer_sizes=(256, 128),
        activation="relu",
        solver="adam",
        alpha=1e-4,
        batch_size=128,
        learning_rate="adaptive",
        max_iter=200,
        random_state=42
    ))
])

mlp_model.fit(X_train, y_train)

mlp_preds = mlp_model.predict(X_test)

print("\n MLP Results:")
print(classification_report(y_test, mlp_preds))
print("Confusion Matrix:\n", confusion_matrix(y_test, mlp_preds))


# --------------------------------------------------------
# 6. Evaluation A: MNIST → MNIST Test
# --------------------------------------------------------

# --------------------------------------------------------
# 7. Evaluation B: MNIST → CIFAR-10 (OOD Test)
# --------------------------------------------------------


print("🔄 Loading MNIST dataset...")
df_cifar = pd.read_csv(DATA_PATH_MNIST)

# Ensure SAME feature schema
X_cifar = df_cifar[numeric_features + categorical_features]
y_cifar = df_cifar["label"].astype(int)

cifar_preds =  mlp_model.predict(X_cifar)

print(classification_report(y_cifar, cifar_preds))
print("Confusion Matrix:\n", confusion_matrix(y_cifar, cifar_preds))


🔹 Loading CIFAR dataset...
🔧 Preparing features...
✂️ Splitting train-test...

🧠 Training MLP...

 MLP Results:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00       875
           1       1.00      0.95      0.97        73
           2       1.00      1.00      1.00        76
           3       1.00      1.00      1.00        77
           4       0.95      1.00      0.97        75
           5       1.00      1.00      1.00        74

    accuracy                           1.00      1250
   macro avg       0.99      0.99      0.99      1250
weighted avg       1.00      1.00      1.00      1250

Confusion Matrix:
 [[875   0   0   0   0   0]
 [  0  69   0   0   4   0]
 [  0   0  76   0   0   0]
 [  0   0   0  77   0   0]
 [  0   0   0   0  75   0]
 [  0   0   0   0   0  74]]
🔄 Loading MNIST dataset...
              precision    recall  f1-score   support

           0       1.00      1.00      1.00      3500
           1       0.96    